# 03 — RAG Preparation


## 1. Install Libraries

`pdfplumber` — extracts text from PDFs cleanly

`sentence-transformers` — converts text to vectors (embeddings)

`chromadb` — local vector database, stores and searches vectors

Why `pdfplumber` over `PyPDF2`?
pdfplumber handles complex layouts better — tables, columns,
headers. Party platforms have complex formatting.

In [ ]:
# !pip install pdfplumber sentence-transformers chromadb -q
!pip install fitz -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

import chromadb
import os
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import fitz
import re


os.makedirs('data/vectordb', exist_ok=True)

PDFS = {
    'democrat':   'data/raw/pdfs/democratic_2024.pdf',
    'republican': 'data/raw/pdfs/republican_2024.pdf',
}

print('Ready')


Ready


## 2. Extract Text From PDFs

**What pdfplumber does:**

Opens each page and extracts the raw text.
Images are ignored automatically — only text is returned.

**What we do with the text:**

Each page becomes one string. We clean it lightly:
- Remove page numbers
- Remove excessive whitespace
- Keep everything else — full paragraphs, arguments, positions

We do NOT remove any political content here.
The more text the RAG has, the better it can answer.

In [ ]:

def extract_pdf_text(pdf_path):
    """
    Extract text from a PDF file page by page.
    Returns list of (page_number, text) tuples.
    """
    pages = []
    party="republican" if "republican" in pdf_path else "democrat"

    doc = fitz.open(pdf_path)

    print(f'Pages: {len(doc)}')

    for i in range(len(doc)):
        text = doc[i].get_text()
        
        if ((i in {0, 1, 2, 9} or i >= 24) and party=="republican") or ((i in {0, 3, 4, 3}) and party=="democrat"):  # Skip first few pages (often TOC, intro, etc.)
            continue

        if not text:
            continue

        # Remove excessive whitespace/newlines
        text = re.sub(r'\s+', ' ', text).strip()

        # Skip pages with very little text
        if len(text.split()) < 30:
            continue

        pages.append((i + 1, text))

    return pages



all_pages = {}

for ideology, path in PDFS.items():
    print(f'\nExtracting {ideology} platform...')

    pages = extract_pdf_text(path)

    all_pages[ideology] = pages

    print(f'Extracted {len(pages)} pages')
    print(f'Sample (page 1): {pages[0][1][:200]}...')


Extracting democrat platform...
Pages: 92
Extracted 88 pages
Sample (page 1): Democratic National Convention Land Acknowledgement The Democratic National Committee wishes to acknowledge that we gather together to state our values on lands that have been stewarded through many c...

Extracting republican platform...
Pages: 28
Extracted 18 pages
Sample (page 1): 4 AMERICA FIRST: AMERICA FIRST: A RETURN TO COMMON SENSE A RETURN TO COMMON SENSE Our Nation’s History is filled with the stories of brave men and women who gave everything they had to build America i...


## 3. Chunking 



In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        # Move forward by (chunk_size - overlap)
        # This creates the overlap
        start += chunk_size - overlap

        # Stop if remaining words are too few to be useful
        if len(words) - start < 50:
            break

    return chunks


# Apply chunking to all pages
all_chunks = []

for ideology, pages in all_pages.items():
    ideology_chunks = 0
    for page_num, text in pages:
        chunks = chunk_text(text, chunk_size=200, overlap=50)
        for chunk_idx, chunk in enumerate(chunks):
            all_chunks.append({
                'text':      chunk,
                'ideology':  ideology,
                'page':      page_num,
                'chunk_idx': chunk_idx,
                # Unique ID for ChromaDB
                'id': f'{ideology}_p{page_num}_c{chunk_idx}'
            })
            ideology_chunks += 1
    print(f'{ideology}: {ideology_chunks} chunks from {len(pages)} pages')

print(f'\nTotal chunks: {len(all_chunks)}')
print(f'Average chunk size: {sum(len(c["text"].split()) for c in all_chunks) // len(all_chunks)} words')

# Show a sample chunk
print('\nSample chunk (democrat):')
dem_chunk = next(c for c in all_chunks if c['ideology'] == 'democrat')
print(dem_chunk['text'][:300])


democrat: 301 chunks from 88 pages
republican: 38 chunks from 18 pages

Total chunks: 339
Average chunk size: 176 words

Sample chunk (democrat):
Democratic National Convention Land Acknowledgement The Democratic National Committee wishes to acknowledge that we gather together to state our values on lands that have been stewarded through many centuries by the ancestors and descendants of Tribal Nations who have been here since time immemorial


## 4. Embedding Model


In [37]:
# Load embedding model
# This downloads ~80MB on first run
print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded')

# Test it — see what a vector looks like
test_text = 'We will build the wall and secure our borders'
test_vector = embedder.encode(test_text)
print(f'\nTest text: {test_text}')
print(f'Vector dimensions: {len(test_vector)}')
print(f'First 10 numbers: {test_vector[:10].round(3)}')
print()
print('These 384 numbers represent the MEANING of that sentence.')
print('Similar sentences will have similar numbers.')


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4930.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded

Test text: We will build the wall and secure our borders
Vector dimensions: 384
First 10 numbers: [-0.022  0.081 -0.005  0.017  0.028 -0.013 -0.037 -0.074 -0.035 -0.048]

These 384 numbers represent the MEANING of that sentence.
Similar sentences will have similar numbers.


## 5. Store in ChromaDB


In [ ]:
print('Initializing ChromaDB...')
client = chromadb.PersistentClient(path='data/vectordb')

# Delete collection if exists (clean start)
try:
    client.delete_collection('platforms')
    print('Deleted existing collection')
except:
    pass


collection = client.create_collection(
    name='platforms',
    metadata={'description': 'Party platform chunks 2024'}
)
print('Collection created')


Initializing ChromaDB...
Deleted existing collection
Collection created


In [ ]:
EMBED_BATCH_SIZE = 64  # embed 64 chunks at a time

print(f'Embedding and storing {len(all_chunks)} chunks...')

for i in tqdm(range(0, len(all_chunks), EMBED_BATCH_SIZE), desc='Embedding'):
    batch = all_chunks[i:i+EMBED_BATCH_SIZE]

    # Extract fields
    texts     = [c['text']     for c in batch]
    ids       = [c['id']       for c in batch]
    metadatas = [{
        'ideology': c['ideology'],
        'page':     c['page'],
    } for c in batch]

    # Convert texts to vectors
    embeddings = embedder.encode(texts, show_progress_bar=False)

    # Store in ChromaDB
    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
        ids=ids
    )

print(f'\nStored {collection.count()} chunks in ChromaDB')


Embedding and storing 339 chunks...


Embedding: 100%|██████████| 6/6 [00:01<00:00,  3.38it/s]


Stored 339 chunks in ChromaDB


## 6. Test Retrieval

simulate exactly what happens
at debate time when an agent needs to retrieve their party position.



In [ ]:
def retrieve_position(topic, ideology, n_results=3):

    query_vector = embedder.encode(topic).tolist()

    # Search ChromaDB
    # where = filter by metadata (ideology)
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results,
        where={'ideology': ideology}  # only this party's platform
    )

    # Combine top chunks into one context string
    chunks = results['documents'][0]
    context = '\n\n'.join(chunks)
    return context


# Test 1: Republican immigration position
print('=== REPUBLICAN position on IMMIGRATION ===')
print(retrieve_position('immigration', 'republican'))
print()

# Test 2: Democrat healthcare position
print('=== DEMOCRAT position on HEALTHCARE ===')
print(retrieve_position('healthcare', 'democrat'))
print()

# Test 3: Republican climate position
print('=== REPUBLICAN position on CLIMATE ===')
print(retrieve_position('climate', 'republican'))


=== REPUBLICAN position on IMMIGRATION ===
History President Trump and Republicans will reverse the Democrats’ destructive Open Borders Policies that have allowed criminal gangs and Illegal Aliens from around the World to roam the United States without consequences. The Republican Party is committed to sending Illegal Aliens back home and removing those who have violated our Laws. 4. Strict Vetting Republicans will use existing Federal Law to keep foreign Christian-hating Communists, Marxists, and Socialists out of America. Those who join our Country must love our Country. We will use extreme vetting to ensure that jihadists and jihadist sympathizers are not admitted. CHAPTER TWO

to impose a full Fentanyl Blockade on the waters of our Region—boarding and inspecting ships to look for fentanyl and fentanyl precursors. Before we defend the Borders of Foreign Countries, we must first secure the Border of our Country. 2. Enforce Immigration Laws Republicans will strengthen ICE, increase pe